# 01 阶段理论推导与成败证明

这个 notebook 只分析 01，不继续推进 02-05。

目标有三件事：

1. 说明 `xref -> GCC/TDOA -> 2D 源位置` 的理论链条。
2. 用 `W1/W2/W3` 的现有实验结果验证什么时候能成功。
3. 证明当前为什么在严格 gate 下失败。

In [ ]:
from pathlib import Path
import json
import sys
from IPython.display import Image, Markdown, display
import matplotlib.pyplot as plt

ROOT = Path.cwd()
for parent in [ROOT] + list(ROOT.parents):
    if (parent / 'python_impl').exists() and (parent / 'README.md').exists():
        ROOT = parent
        break
if str(ROOT / 'python_impl') not in sys.path:
    sys.path.insert(0, str(ROOT / 'python_impl'))

from experiments.hypothesis_validation.01_source_localization_anechoic_2d.src.analysis_utils import (
    WINDOW_KEYS,
    compute_sensitivity_stats,
    extract_position_metric_rows,
    extract_tdoa_metric_rows,
    load_all_run_bundles,
    make_verdict,
)

runs = load_all_run_bundles()
position_plain = extract_position_metric_rows(runs, method='plain_gcc')
position_phat = extract_position_metric_rows(runs, method='gcc_phat')
tdoa_plain = extract_tdoa_metric_rows(runs, method='plain_gcc', scope='overall')
tdoa_phat = extract_tdoa_metric_rows(runs, method='gcc_phat', scope='overall')
sensitivity = compute_sensitivity_stats(window_key='W3', method='gcc_phat')
verdict = make_verdict(runs)
step_dir = ROOT / 'python_impl' / 'experiments' / 'hypothesis_validation' / '01_source_localization_anechoic_2d'
report_path = step_dir / 'results' / 'stage01_theory_and_failure_analysis_zh.md'
print('ROOT =', ROOT)
print('report =', report_path)
print(json.dumps(verdict, ensure_ascii=False, indent=2))


## 理论推导

在 01 的最简条件下，我们只考虑直达声、二维平面、固定声速 `c`、单个白噪声源。

第 `i` 个参考麦的观测模型为

$$
x_i[n] = a_i\, s[n-\tau_i],\qquad \tau_i = \frac{\|p-r_i\|}{c}
$$

其中 `p` 是源位置，`r_i` 是第 `i` 个参考麦位置。

对白噪声源，两通道互功率谱可以近似写成

$$
S_{ij}(\omega) = A_{ij}(\omega) e^{-j\omega(\tau_i-\tau_j)}
$$

做 PHAT 归一化后只保留相位项，因此逆变换峰值位置对应的就是 TDOA

$$
\Delta\tau_{ij} = \tau_i - \tau_j
$$

这一步恢复的是三对参考麦的时延差，而不是直接恢复位置。

位置反演来自双曲线约束：

$$
\|p-r_i\| - \|p-r_j\| = c\,\Delta\tau_{ij}
$$

把三对麦 `(01, 02, 12)` 组合起来，就得到一个带房间边界的最小二乘问题。

定义

$$
g(p) = [d_{01}(p), d_{02}(p), d_{12}(p)]^\top
$$

则局部误差传播满足

$$
\delta g \approx J(p)\,\delta p,
\qquad
\delta p \approx J(p)^+\,\delta g
$$

因此成功不仅要求 TDOA 误差小，还要求 `J` 的条件数不能太坏。少数几何病态样本下，小的 TDOA 误差会被放大成很大的位置误差。

In [ ]:
display(Markdown('## 关键数值复现'))
for window_key in WINDOW_KEYS:
    summary = runs[window_key]['summary']
    phat = summary['analytic_gcc_phat']
    print(f'[{window_key}] GCC-PHAT')
    print('  iid_test : median = %.6f m, p90 = %.6f m' % (phat['iid_test']['median_m'], phat['iid_test']['p90_m']))
    print('  geom_test: median = %.6f m, p90 = %.6f m' % (phat['geom_test']['median_m'], phat['geom_test']['p90_m']))
    print('  tdoa p90 : iid = %.6f samples, geom = %.6f samples' % (
        phat['iid_test']['delay_error']['overall']['p90_samples'],
        phat['geom_test']['delay_error']['overall']['p90_samples'],
    ))
    print()

print('W3 sensitivity summary:')
print(json.dumps(sensitivity, ensure_ascii=False, indent=2))


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
splits = ('iid_test', 'geom_test')
for row_group, metric_name in enumerate(('median_m', 'p90_m')):
    for col, split_name in enumerate(splits):
        ax = axes[row_group, col]
        for method_name, rows, color in [
            ('plain GCC', position_plain, 'tab:purple'),
            ('GCC-PHAT', position_phat, 'tab:orange'),
        ]:
            xs = []
            ys = []
            for idx, window_key in enumerate(WINDOW_KEYS):
                match = next(row for row in rows if row['window_key'] == window_key and row['split'] == split_name)
                xs.append(idx)
                ys.append(match[metric_name])
            ax.plot(xs, ys, marker='o', label=method_name, color=color)
        ax.set_xticks(range(len(WINDOW_KEYS)), WINDOW_KEYS)
        ax.set_title(f'{split_name} | {metric_name}')
        ax.set_ylabel('meters')
        ax.grid(True, alpha=0.25)
        ax.legend()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for col, split_name in enumerate(('iid_test', 'geom_test')):
    ax = axes[col]
    for method_name, rows, color in [
        ('plain GCC', tdoa_plain, 'tab:purple'),
        ('GCC-PHAT', tdoa_phat, 'tab:orange'),
    ]:
        xs = []
        ys = []
        for idx, window_key in enumerate(WINDOW_KEYS):
            match = next(row for row in rows if row['window_key'] == window_key and row['split'] == split_name)
            xs.append(idx)
            ys.append(match['p90_samples'])
        ax.plot(xs, ys, marker='o', label=method_name, color=color)
    ax.set_xticks(range(len(WINDOW_KEYS)), WINDOW_KEYS)
    ax.set_title(f'{split_name} | TDOA p90')
    ax.set_ylabel('samples')
    ax.grid(True, alpha=0.25)
    ax.legend()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
labels = ['worst 10%', 'rest 90%']
for ax, split_name in zip(axes, ('iid_test', 'geom_test')):
    split_stats = sensitivity[split_name]
    area_vals = [split_stats['worst10_area_mean'], split_stats['rest90_area_mean']]
    cond_vals = [split_stats['worst10_cond_mean'], split_stats['rest90_cond_mean']]
    x = [0, 1]
    ax.bar([v - 0.15 for v in x], area_vals, width=0.3, label='triangle area', color='tab:blue')
    ax.bar([v + 0.15 for v in x], cond_vals, width=0.3, label='Jacobian cond', color='tab:red')
    ax.set_xticks(x, labels)
    ax.set_title(split_name)
    ax.grid(True, axis='y', alpha=0.25)
    ax.legend()
plt.show()

print('Worst examples from W3 + GCC-PHAT:')
for split_name in ('iid_test', 'geom_test'):
    print(f'[{split_name}]')
    for row in sensitivity[split_name]['worst_examples']:
        print(row)
    print()


In [ ]:
plots_dir = runs['W3']['paths']['plots_dir']
display(Markdown('## W3 典型成功 / 失败样本'))
display(Image(filename=str(plots_dir / 'analytic_gcc_phat_success.png')))
display(Image(filename=str(plots_dir / 'analytic_gcc_phat_failure.png')))
display(Image(filename=str(plots_dir / 'analytic_compare_iid_test.png')))
display(Image(filename=str(plots_dir / 'analytic_compare_geom_test.png')))


## 最终结论

- 当前 01 在中位数意义上是可行的，因为 `W1/W2/W3` 的 `gcc_phat` 中位误差始终约为 `7 mm`。
- 但它在当前严格 gate 下失败，因为 `iid_test` 与 `geom_test` 的 `p90` 始终没有过线。
- TDOA `p90` 只有约 `0.122 samples`，说明延迟估计精度已经足够高。
- 最差 `10%` 样本对应更小的参考麦三角形面积和更高的 Jacobian 条件数，因此失败主要来自几何病态，而不是来自 `xref` 长度不足。

**正式判决：`01 failed`。**

在重审 01 的几何条件和目标判据之前，不应继续推进 02-05。